In [1]:
from pathlib import Path
import pandas as pd

RESULTS = Path.cwd().parent / "results"
agent = pd.read_csv(RESULTS / "agent_dev.csv")
base = pd.read_csv(RESULTS / "baseline_dev.csv")
df = agent.merge(base[["log", "hit"]], on="log", suffixes=("", "_base"))
pd.crosstab(df["hit_base"], df["hit"], rownames=["baseline"], colnames=["agent"])

# hits where the answer was / wasn't in the last 150 lines
df.groupby("in_tail")[["hit", "hit_base"]].sum()

# logs the baseline got and the agent missed: read the trace to see what it did
losses = df[df["hit_base"] & ~df["hit"]]
losses[["repo", "start_line", "end_line", "predicted_start", "predicted_end", "steps", "trace"]]


,repo,start_line,end_line,predicted_start,predicted_end,steps,trace
79,git@git,3251,3252,3261,3261,2,"[{""tool"": ""search_log"", ""arguments"": {""pattern..."
91,powershell@PowerShell,1044,1054,1101,1115,0,[]
97,git@git,3249,3250,3314,3318,5,"[{""tool"": ""read_lines"", ""arguments"": {""start"":..."
107,git@git,3251,3252,0,0,2,"[{""tool"": ""search_log"", ""arguments"": {""pattern..."
135,git@git,3249,3250,3314,3318,5,"[{""tool"": ""read_lines"", ""arguments"": {""end"": 3..."
152,git@git,3249,3250,3294,3318,5,"[{""tool"": ""read_lines"", ""arguments"": {""start"":..."
168,avelino@awesome-go,4625,4626,0,0,2,"[{""tool"": ""search_log"", ""arguments"": {""pattern..."
179,git@git,3251,3252,0,0,2,"[{""tool"": ""search_log"", ""arguments"": {""pattern..."
185,FreeCodeCamp@freecodecamp,1029,1029,1030,1055,5,"[{""tool"": ""read_lines"", ""arguments"": {""start"":..."
192,vsouza@awesome-ios,3353,3372,3403,3417,3,"[{""tool"": ""read_lines"", ""arguments"": {""start"":..."


In [2]:
from math import comb
from pathlib import Path
import pandas as pd

SPLIT = "eval"   # "dev" to look at the practice runs
RESULTS = Path.cwd().parent / "results"
agent = pd.read_csv(RESULTS / f"agent_{SPLIT}.csv")
base = pd.read_csv(RESULTS / f"baseline_{SPLIT}.csv")
df = agent.merge(base[["log", "hit"]], on="log", suffixes=("", "_base"))
pd.crosstab(df["hit_base"], df["hit"], rownames=["baseline"], colnames=["agent"])

# exact McNemar test: only the logs where the two disagree count
wins = int((~df["hit_base"] & df["hit"]).sum())
losses = int((df["hit_base"] & ~df["hit"]).sum())
n = wins + losses
p = min(1, 2 * sum(comb(n, i) for i in range(min(wins, losses) + 1)) / 2**n)
wins, losses, round(p, 4)

df.groupby("in_tail")[["hit", "hit_base"]].agg(["sum", "count"])

df[["seconds", "prompt_tokens", "steps"]].describe().round(1)

,seconds,prompt_tokens,steps
count,180.0,180.0,180.0
mean,27.0,2709.6,3.5
std,27.4,1905.7,2.2
min,3.7,291.0,0.0
25%,15.5,1733.2,2.0
50%,20.9,2211.5,3.0
75%,30.1,3062.0,4.0
max,306.9,15700.0,8.0
